In [ ]:
# STEP 1 — Setup imports and file paths
# Description: Import required libraries and define all input/output paths

import pandas as pd
import numpy as np
from pathlib import Path

# File paths
epc_file = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"
schedules_file = "/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/schedules.csv"
base_models_dir = Path("/Users/jakegehrung/Desktop/data_raw/baseline_models")

In [2]:
# STEP 2 — Load schedules and build half-hour time template
# Description: Read weekday/weekend schedules and create 48-step half-hour time labels

schedules_df = pd.read_csv(schedules_file)

# Ensure expected columns exist
assert "lighting_wd" in schedules_df.columns
assert "lighting_we" in schedules_df.columns

# Extract schedules as arrays (assumes 48 half-hour rows in correct order)
wd_schedule = schedules_df["lighting_wd"].values
we_schedule = schedules_df["lighting_we"].values

# Create half-hour time labels starting at 00:30 ending at 00:00
time_labels = []
hour = 0
minute = 30

for i in range(48):
    time_labels.append(f"{hour:02d}:{minute:02d}")
    
    minute += 30
    if minute == 60:
        minute = 0
        hour += 1
    if hour == 24:
        hour = 0

# Safety check
assert len(time_labels) == 48
assert len(wd_schedule) == 48
assert len(we_schedule) == 48

In [3]:
# STEP 3 — Load EPC building IDs
# Description: Extract all BUILDING_REFERENCE_NUMBER values from EPC dataset

epc_df = pd.read_csv(epc_file)

assert "BUILDING_REFERENCE_NUMBER" in epc_df.columns

building_ids = epc_df["BUILDING_REFERENCE_NUMBER"].dropna().astype(str).unique()

len(building_ids), building_ids[:5]

(117,
 array(['1001829067', '1001951858', '1001829071', '1234761001',
        '1001991633'], dtype=object))

In [4]:
# STEP 4 — Define function to create full-year lighting schedule
# Description: Generates 2026 half-hourly schedule and writes CSV into each LIGHTING folder

def create_lighting_schedule(building_id: str):
    lighting_folder = base_models_dir / building_id / "LIGHTING"
    lighting_folder.mkdir(parents=True, exist_ok=True)
    
    output_file = lighting_folder / "lighting_schedule.csv"
    
    # Build full 2026 calendar (non-leap year assumed)
    dates = pd.date_range("2026-01-01", "2026-12-31", freq="D")
    
    all_rows = []
    
    for date in dates:
        is_weekend = date.weekday() >= 5  # 5=Saturday, 6=Sunday
        
        schedule = we_schedule if is_weekend else wd_schedule
        
        for t, frac in zip(time_labels, schedule):
            all_rows.append({
                "Time": t,
                "lighting_fraction": frac
            })
    
    df_out = pd.DataFrame(all_rows)
    
    # Overwrite if exists
    df_out.to_csv(output_file, index=False)

In [5]:
# STEP 5 — Execute for all buildings
# Description: Loop through all building IDs and generate lighting_schedule.csv

for b_id in building_ids:
    try:
        create_lighting_schedule(b_id)
    except Exception as e:
        print(f"Error processing {b_id}: {e}")

print("Lighting schedules generated for all buildings.")

Lighting schedules generated for all buildings.
